In [1]:
import pandas as pd
from pathlib import Path

processed_path = Path("../../data/processed")

train_data = pd.read_parquet(processed_path / "ml_train.parquet")
test_data = pd.read_parquet(processed_path / "ml_test.parquet")

X_train = train_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_train = train_data["HOSPITAL_EXPIRE_FLAG"]
subject_id_train = train_data["SUBJECT_ID"]

X_test = test_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_test = test_data["HOSPITAL_EXPIRE_FLAG"]

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

categorical_cols = [
    "gender",
    "admission_type",
    "admission_location"
]

numeric_cols = X_train.columns.difference(categorical_cols).tolist()

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [3]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    0: weights[0],
    1: weights[1]
}

final_model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train_processed.shape[1],)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

final_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print(class_weights)

I0000 00:00:1789563847.821381   30828 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789563847.822128   30828 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789563848.246568   30828 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789563849.932354   30828 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

{0: np.float64(0.5679448527102569), 1: np.float64(4.179454587474001)}


/home/sengul/projects/icu-risk-prediction/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1789563852.344953   30828 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1789563852.345393   32468 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1789563852.388115   30828 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if

In [4]:
history = final_model.fit(
    X_train_processed,
    y_train,
    epochs=8,
    batch_size=64,
    class_weight=class_weights,
    verbose=1
)

Epoch 1/8
566/566 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7357 - loss: 0.5105
Epoch 2/8
566/566 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7649 - loss: 0.4562
Epoch 3/8
566/566 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7715 - loss: 0.4364
Epoch 4/8
566/566 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7811 - loss: 0.4222
Epoch 5/8
566/566 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7889 - loss: 0.4103
Epoch 6/8
566/566 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7900 - loss: 0.3985
Epoch 7/8
566/566 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7994 - loss: 0.3868
Epoch 8/8
566/566 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8049 - loss: 0.3756


In [5]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

test_probs = final_model.predict(X_test_processed).ravel()

test_pred = (test_probs >= 0.6).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, test_probs))
print("Precision:", precision_score(y_test, test_pred))
print("Recall:", recall_score(y_test, test_pred))
print("F1:", f1_score(y_test, test_pred))

284/284 ━━━━━━━━━━━━━━━━━━━━ 0s 819us/step
ROC-AUC: 0.8519335172447957
Precision: 0.3472906403940887
Recall: 0.6824782187802517
F1: 0.4603330068560235


## Final Test Evaluation

The selected Deep Learning model was evaluated on the untouched test set using the previously selected threshold of `0.6`.

Final test performance:

- ROC-AUC: **0.852**
- Precision: **0.347**
- Recall: **0.682**
- F1: **0.460**

Compared with validation results, performance decreased slightly, but the model maintained strong recall and similar overall discrimination.

The final model successfully preserved the recall-focused objective on unseen test data.